In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [20]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
!wget $data -O data-week-3.csv

--2026-04-27 20:08:00--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  --.-KB/s    in 0.05s   

2026-04-27 20:08:00 (19.8 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [21]:
df = pd.read_csv('data-week-3.csv')

In [22]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [23]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Data Standardisation

In [24]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
strings = list(df.dtypes[df.dtypes == 'object'].index)

for col in strings:
  df[col] = df[col].str.lower().str.replace(' ', '_')


In [25]:
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [26]:
df.totalcharges = pd.to_numeric(df['totalcharges'], errors='coerce')

In [27]:
df.totalcharges = df.totalcharges.fillna(0)

In [28]:
df.churn = (df.churn == 'yes').astype(int)


# Setting up validation framework

In [29]:
from sklearn.model_selection import train_test_split

In [30]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [31]:
df_train, df_val = train_test_split(df, test_size=0.25, random_state=1)

In [32]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [33]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values


In [34]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

# Exploratory Data Analysis

In [46]:
global_churn = round(df_full_train.churn.mean(),2)
global_churn

np.float64(0.27)

In [47]:
numericals = ['tenure', 'monthlycharges', 'totalcharges']

categoricals = ['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'churn']

In [50]:
df_full_train[categoricals].nunique()

,0
customerid,5634
gender,2
seniorcitizen,2
partner,2
dependents,2
phoneservice,2
multiplelines,3
internetservice,3
onlinesecurity,3
onlinebackup,3


In [51]:
from IPython.display import display

# Feature importance:

In [54]:
for c in categoricals:
  print(c)
  df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
  df_group['diff'] = df['mean'] - global_churn
  df_group['risk'] = df['mean'] / global_churn

  display(df_group)
  print()
  print()

customerid


,mean,count,diff,risk
customerid,,,,
0002-orfbo,0.0,1,NaN,NaN
0004-tlhlj,1.0,1,NaN,NaN
0011-igkff,1.0,1,NaN,NaN
0013-exchz,1.0,1,NaN,NaN
0013-mhzwf,0.0,1,NaN,NaN
...,...,...,...,...
9987-lutyd,0.0,1,NaN,NaN
9992-rramn,1.0,1,NaN,NaN
9992-ujoel,0.0,1,NaN,NaN




gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,-0.000791,0.997069
male,0.263214,2838,-0.008397,0.968901




seniorcitizen


...

,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,NaN,NaN
yes,0.338151,3321,NaN,NaN




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,NaN,NaN
credit_card_(automatic),0.164339,1217,NaN,NaN
electronic_check,0.455890,1893,NaN,NaN
mailed_check,0.193870,1305,NaN,NaN




churn


,mean,count,diff,risk
churn,,,,
0,0.0,4113,NaN,NaN
1,1.0,1521,NaN,NaN
